# RNA-seq Analysis: From Reads to Differential Expression

RNA-seq (RNA sequencing) is the dominant technology for studying gene expression at the transcriptome level. This notebook covers the complete RNA-seq analysis workflow, from experimental design through differential expression and visualization.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Describe the RNA-seq experimental workflow and key design decisions
2. Understand alignment-based and alignment-free quantification strategies
3. Work with count matrices and apply normalization methods
4. Perform basic differential expression analysis
5. Interpret results using volcano plots, MA plots, and gene set enrichment
6. Implement a complete mini-workflow with simulated data

---

[← Previous: Variant Calling and SNP Analysis](../../Tier_3_Applied_Bioinformatics/02_Variant_Calling_and_SNP_Analysis/01_variant_calling_and_snp_analysis.ipynb) | [Next: Microbial Diversity Analysis →](../../Tier_3_Applied_Bioinformatics/04_Microbial_Diversity/01_microbial_diversity.ipynb)

---

## 1. Why RNA-seq?

Before RNA-seq, gene expression was measured with **microarrays**, which rely on hybridization to pre-designed probes. RNA-seq offers several advantages:

| Feature | Microarrays | RNA-seq |
|---------|-------------|--------|
| Dynamic range | ~3 orders of magnitude | ~5+ orders of magnitude |
| Prior knowledge | Requires probe design | No reference needed (de novo) |
| Novel transcripts | Cannot detect | Can discover new isoforms, fusions |
| Quantification | Relative (intensity) | Digital (read counts) |
| Background noise | Cross-hybridization | Low, especially with proper QC |
| Cost per sample | Low | Medium-high (decreasing) |

### Common Applications

- **Differential gene expression**: Compare conditions (e.g., tumor vs. normal)
- **Transcript isoform discovery**: Alternative splicing, novel transcripts
- **Allele-specific expression**: Which allele is more active?
- **Single-cell RNA-seq (scRNA-seq)**: Expression at cellular resolution
- **Spatial transcriptomics**: Gene expression in tissue context

## 2. Experimental Design

Good experimental design is critical. Mistakes here cannot be fixed computationally.

### Key Decisions

**Biological replicates**: At least 3 per condition, ideally 5+. More replicates improve statistical power far more than deeper sequencing of fewer samples.

**Sequencing depth**: For differential expression in well-annotated organisms, 10-30 million reads per sample is standard. For transcript discovery or lowly-expressed genes, 50-100M+ may be needed.

**Read length and type**:
- Single-end (SE) 50-75 bp: sufficient for gene-level DE
- Paired-end (PE) 100-150 bp: better for isoform analysis, novel transcript discovery

**Strandedness**: Strand-specific protocols (e.g., dUTP method) preserve which DNA strand was transcribed. This is now standard and helps resolve overlapping genes on opposite strands.

### Library Preparation

```
Total RNA
    |
    v
RNA selection (poly-A selection or rRNA depletion)
    |
    v
Fragmentation (chemical or enzymatic)
    |
    v
Reverse transcription -> cDNA
    |
    v
Adapter ligation
    |
    v
PCR amplification (limited cycles)
    |
    v
Sequencing (Illumina, etc.)
```

**poly-A selection** enriches for mRNA (coding genes); **rRNA depletion** retains non-coding RNA and is needed for degraded samples (e.g., FFPE tissue).

### Batch Effects

Process all samples together when possible. If batches are unavoidable, balance conditions across batches. Record batch information for statistical correction later.

## 3. RNA-seq Workflow Overview

```
FASTQ files (raw reads)
      |
      v
Quality Control (FastQC, Trim Galore, fastp)
      |
      v
  +---+---+
  |       |
  v       v
Alignment-based          Alignment-free
(STAR/HISAT2)            (Salmon/kallisto)
  |                           |
  v                           v
BAM files              Transcript quantifications
  |                           |
  v                           |
featureCounts/HTSeq           |
  |                           |
  +-------+-------------------+
          |
          v
   Count matrix (genes x samples)
          |
          v
   Normalization
          |
          v
   Differential expression (DESeq2 / edgeR)
          |
          v
   Visualization & interpretation
          |
          v
   Gene set enrichment (GSEA / ORA)
```

## 4. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.decomposition import PCA
from collections import OrderedDict

np.random.seed(42)

# Plotting defaults
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

## 5. Alignment-Based Quantification

### Step 1: Alignment

Splice-aware aligners handle reads that span exon-exon junctions.

| Aligner | Speed | Memory | Notes |
|---------|-------|--------|-------|
| **STAR** | Very fast | 30+ GB RAM | Most widely used for human/mouse |
| **HISAT2** | Fast | ~8 GB RAM | Successor to TopHat2, lower memory |

```bash
# STAR: build genome index (once)
STAR --runMode genomeGenerate \
     --genomeDir star_index/ \
     --genomeFastaFiles genome.fa \
     --sjdbGTFfile genes.gtf

# STAR: align reads
STAR --genomeDir star_index/ \
     --readFilesIn sample_R1.fastq sample_R2.fastq \
     --outSAMtype BAM SortedByCoordinate \
     --quantMode GeneCounts
```

### Step 2: Counting reads per gene

After alignment, we count how many reads map to each gene using the gene annotation (GTF file).

| Tool | Approach | Speed |
|------|----------|-------|
| **featureCounts** (Subread) | Assigns reads to features by overlap | Very fast |
| **HTSeq-count** | Assigns reads, strict handling of ambiguity | Slower, more conservative |

```bash
# featureCounts example
featureCounts -a genes.gtf \
              -o counts.txt \
              -T 4 \
              -p --countReadPairs \
              sample1.bam sample2.bam sample3.bam
```

The result is a **count matrix**: rows are genes, columns are samples, values are integer read counts.

## 6. Alignment-Free Quantification

Alignment-free methods skip the computationally expensive genome alignment step. Instead, they quantify transcript abundance directly from reads using lightweight algorithms (pseudo-alignment or quasi-mapping).

| Tool | Algorithm | Speed | Output |
|------|-----------|-------|--------|
| **Salmon** | Quasi-mapping + EM | ~minutes per sample | TPM per transcript |
| **kallisto** | Pseudo-alignment + EM | ~minutes per sample | TPM per transcript |

```bash
# Salmon: build transcriptome index
salmon index -t transcriptome.fa -i salmon_index

# Salmon: quantify
salmon quant -i salmon_index \
             -l A \
             -1 sample_R1.fastq \
             -2 sample_R2.fastq \
             -o sample_quant \
             --validateMappings
```

### Key Advantage: Speed and Accuracy

Salmon/kallisto are **10-100x faster** than alignment-based methods while providing comparable or better accuracy for transcript-level quantification.

### From Transcripts to Genes: tximport

Salmon and kallisto output transcript-level estimates. The R package `tximport` aggregates these to gene level for use with DESeq2/edgeR, properly handling the uncertainty in transcript assignment. In Python, this can be done with `pydeseq2` or manually summing counts per gene.

## 7. Expression Units: RPKM, FPKM, TPM

Raw counts are not directly comparable between genes (different lengths) or between samples (different sequencing depths). Several normalization units exist:

### RPKM (Reads Per Kilobase per Million mapped reads)

$$\text{RPKM}_i = \frac{C_i}{L_i \times N} \times 10^9$$

where $C_i$ = read count for gene $i$, $L_i$ = gene length in bp, $N$ = total mapped reads.

### FPKM (Fragments Per Kilobase per Million)

Same as RPKM but for paired-end data: counts *fragments* (read pairs) instead of individual reads.

### TPM (Transcripts Per Million)

$$\text{TPM}_i = \frac{C_i / L_i}{\sum_j C_j / L_j} \times 10^6$$

TPM normalizes by gene length first, then by total. **TPM values sum to 1 million in every sample**, making samples directly comparable. TPM is now the preferred unit.

### Why Not Use These for DE?

RPKM/FPKM/TPM are useful for visualization and reporting, but **not for differential expression testing**. DE tools like DESeq2 and edgeR work with raw counts and apply their own normalization that accounts for library composition bias.

### Practical: Computing TPM from Counts

In [ ]:
# Example: compute TPM from raw counts
gene_names = ['BRCA1', 'TP53', 'MYC', 'GAPDH', 'ACTB', 'IL6', 'TNF', 'EGFR']
gene_lengths = np.array([7088, 2512, 4149, 1310, 1761, 1125, 1569, 5616])  # in bp
raw_counts = np.array([320, 890, 1560, 15200, 12400, 45, 78, 520])

def counts_to_tpm(counts, lengths):
    """Convert raw counts to TPM."""
    rate = counts / lengths  # normalize by gene length
    return rate / rate.sum() * 1e6  # scale to million

tpm = counts_to_tpm(raw_counts, gene_lengths)

tpm_df = pd.DataFrame({
    'Gene': gene_names,
    'Length (bp)': gene_lengths,
    'Raw Counts': raw_counts,
    'TPM': np.round(tpm, 1)
}).set_index('Gene')

print(tpm_df)
print(f"\nTPM sum: {tpm.sum():.0f} (should be 1,000,000)")

## 8. The Count Matrix

The central data structure in RNA-seq analysis is the **count matrix** (or expression matrix):

- Rows = genes (or transcripts)
- Columns = samples
- Values = integer read counts

Let's simulate a realistic count matrix for our downstream analysis.

In [ ]:
def simulate_rnaseq_counts(n_genes=2000, n_samples_per_group=4, n_de_genes=200,
                           base_mean_range=(10, 5000), fold_change_range=(1.5, 4.0)):
    """
    Simulate RNA-seq count data with known differentially expressed genes.
    
    Uses a negative binomial distribution, which models the overdispersion
    typically observed in RNA-seq count data.
    """
    n_samples = n_samples_per_group * 2
    
    # Gene names
    gene_names = [f'Gene_{i:04d}' for i in range(n_genes)]
    
    # Sample names and condition labels
    sample_names = ([f'Control_{i+1}' for i in range(n_samples_per_group)] +
                    [f'Treatment_{i+1}' for i in range(n_samples_per_group)])
    conditions = (['Control'] * n_samples_per_group +
                  ['Treatment'] * n_samples_per_group)
    
    # Base mean expression for each gene (log-normal distribution)
    base_means = np.exp(np.random.uniform(
        np.log(base_mean_range[0]),
        np.log(base_mean_range[1]),
        n_genes
    ))
    
    # Dispersion parameter (inversely related to mean, as in real RNA-seq)
    dispersions = 0.1 + 1.0 / np.sqrt(base_means)
    
    # Generate counts using negative binomial
    counts = np.zeros((n_genes, n_samples), dtype=int)
    true_de = np.zeros(n_genes, dtype=bool)
    true_log2fc = np.zeros(n_genes)
    
    # Select DE genes
    de_indices = np.random.choice(n_genes, n_de_genes, replace=False)
    true_de[de_indices] = True
    
    # Assign fold changes (half up, half down)
    fold_changes = np.ones(n_genes)
    for idx in de_indices[:n_de_genes // 2]:
        fc = np.random.uniform(fold_change_range[0], fold_change_range[1])
        fold_changes[idx] = fc
        true_log2fc[idx] = np.log2(fc)
    for idx in de_indices[n_de_genes // 2:]:
        fc = np.random.uniform(fold_change_range[0], fold_change_range[1])
        fold_changes[idx] = 1.0 / fc
        true_log2fc[idx] = -np.log2(fc)
    
    for j in range(n_samples):
        # Library size factor (slight variation between samples)
        size_factor = np.random.uniform(0.8, 1.2)
        
        for i in range(n_genes):
            mean_expr = base_means[i] * size_factor
            if conditions[j] == 'Treatment':
                mean_expr *= fold_changes[i]
            
            # Negative binomial parameterization
            p = dispersions[i] / (dispersions[i] + mean_expr)
            n_param = dispersions[i] / p if p > 0 else 1
            counts[i, j] = np.random.negative_binomial(
                max(1, int(n_param)), max(1e-10, min(1 - 1e-10, p))
            )
    
    count_df = pd.DataFrame(counts, index=gene_names, columns=sample_names)
    sample_info = pd.DataFrame({'condition': conditions}, index=sample_names)
    gene_info = pd.DataFrame({
        'true_de': true_de,
        'true_log2fc': true_log2fc
    }, index=gene_names)
    
    return count_df, sample_info, gene_info

# Generate simulated data
counts_df, sample_info, gene_info = simulate_rnaseq_counts()

print(f"Count matrix shape: {counts_df.shape} (genes x samples)")
print(f"\nSample info:")
print(sample_info)
print(f"\nDE genes in simulation: {gene_info['true_de'].sum()} out of {len(gene_info)}")
print(f"\nFirst few rows of count matrix:")
counts_df.head(10)

In [ ]:
# Distribution of raw counts (log scale)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram of all counts
all_counts = counts_df.values.flatten()
axes[0].hist(all_counts[all_counts > 0], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Read Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Raw Counts')
axes[0].set_xscale('log')

# Library sizes
lib_sizes = counts_df.sum(axis=0)
colors = ['steelblue' if c == 'Control' else 'coral' for c in sample_info['condition']]
axes[1].bar(range(len(lib_sizes)), lib_sizes / 1e6, color=colors, edgecolor='black')
axes[1].set_xlabel('Sample')
axes[1].set_ylabel('Library Size (millions of reads)')
axes[1].set_title('Library Sizes')
axes[1].set_xticks(range(len(lib_sizes)))
axes[1].set_xticklabels(lib_sizes.index, rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 9. Normalization

Raw counts cannot be compared directly between samples due to:
1. **Library size differences**: Some samples have more total reads
2. **Library composition bias**: A few highly expressed genes can consume a disproportionate share of reads

### Common Normalization Methods

| Method | Used by | Approach |
|--------|---------|----------|
| **DESeq2 median-of-ratios** | DESeq2 | Median ratio to geometric mean across samples |
| **TMM** (Trimmed Mean of M-values) | edgeR | Trimmed mean of log fold changes |
| **CPM** (Counts Per Million) | General | Simple library size normalization |
| **Upper quartile** | General | Normalize to 75th percentile |

### DESeq2 Size Factors

The DESeq2 method computes a **size factor** for each sample:

1. For each gene, compute the geometric mean across all samples
2. For each sample, divide each gene's count by this geometric mean
3. The size factor for the sample is the **median** of these ratios

This is robust to highly expressed genes because the median is not affected by outliers.

In [ ]:
def deseq2_size_factors(count_matrix):
    """
    Compute DESeq2-style size factors (median-of-ratios method).
    """
    # Step 1: geometric mean of each gene across samples
    # (use only genes with non-zero counts in all samples)
    nonzero_mask = (count_matrix > 0).all(axis=1)
    filtered = count_matrix.loc[nonzero_mask]
    
    log_means = np.log(filtered).mean(axis=1)
    geo_means = np.exp(log_means)
    
    # Step 2: for each sample, compute ratio of each gene to its geometric mean
    # Step 3: take the median ratio as the size factor
    size_factors = {}
    for sample in filtered.columns:
        ratios = filtered[sample] / geo_means
        size_factors[sample] = np.median(ratios)
    
    return pd.Series(size_factors)


size_factors = deseq2_size_factors(counts_df)
print("DESeq2 size factors:")
print(size_factors.round(3))
print(f"\nSize factors close to 1.0 indicate samples have similar library composition.")

In [ ]:
# Apply normalization
normalized_counts = counts_df.div(size_factors, axis=1)

# Compare raw vs normalized
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplots of log2 counts per sample
log_raw = np.log2(counts_df + 1)
log_norm = np.log2(normalized_counts + 1)

bp1 = axes[0].boxplot([log_raw[col] for col in log_raw.columns],
                       patch_artist=True, showfliers=False)
for i, patch in enumerate(bp1['boxes']):
    patch.set_facecolor('steelblue' if i < 4 else 'coral')
axes[0].set_xticklabels(log_raw.columns, rotation=45, ha='right')
axes[0].set_ylabel('log2(count + 1)')
axes[0].set_title('Before Normalization')

bp2 = axes[1].boxplot([log_norm[col] for col in log_norm.columns],
                       patch_artist=True, showfliers=False)
for i, patch in enumerate(bp2['boxes']):
    patch.set_facecolor('steelblue' if i < 4 else 'coral')
axes[1].set_xticklabels(log_norm.columns, rotation=45, ha='right')
axes[1].set_ylabel('log2(normalized count + 1)')
axes[1].set_title('After DESeq2 Normalization')

plt.tight_layout()
plt.show()

## 10. Exploratory Data Analysis

Before differential expression testing, we should explore our data to check for outliers, batch effects, and overall sample relationships.

### PCA of Samples

PCA (Principal Component Analysis) reduces the high-dimensional gene expression data to a few principal components that capture the most variance. We expect samples to cluster by condition.

In [ ]:
# PCA on log-transformed normalized counts
log_data = np.log2(normalized_counts + 1)

pca = PCA(n_components=2)
pca_result = pca.fit_transform(log_data.T)  # samples as rows

fig, ax = plt.subplots(figsize=(8, 6))

for condition, color, marker in [('Control', 'steelblue', 'o'), ('Treatment', 'coral', 's')]:
    mask = sample_info['condition'] == condition
    ax.scatter(pca_result[mask, 0], pca_result[mask, 1],
               c=color, marker=marker, s=100, edgecolors='black',
               label=condition, zorder=3)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('PCA of RNA-seq Samples')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Sample-to-Sample Correlation Heatmap

In [ ]:
# Correlation between samples
corr_matrix = log_data.corr(method='spearman')

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr_matrix, cmap='RdYlBu_r', vmin=0.9, vmax=1.0)

ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right')
ax.set_yticklabels(corr_matrix.columns)
ax.set_title('Sample-to-Sample Spearman Correlation')

# Add correlation values
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        ax.text(j, i, f'{corr_matrix.iloc[i, j]:.3f}',
                ha='center', va='center', fontsize=8)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 11. Differential Expression Analysis

### The Statistical Model

RNA-seq count data is modeled with a **negative binomial (NB) distribution**, which accounts for both the Poisson-like counting noise and biological variability (overdispersion).

$$K_{ij} \sim \text{NB}(\mu_{ij}, \alpha_i)$$

where:
- $K_{ij}$ = count for gene $i$ in sample $j$
- $\mu_{ij}$ = mean expression (depends on condition and size factor)
- $\alpha_i$ = dispersion parameter for gene $i$

DESeq2 fits this model and uses a **Wald test** or **likelihood ratio test** to test for differential expression between conditions.

### Key Steps in DESeq2

1. Estimate size factors (normalization)
2. Estimate gene-wise dispersions
3. Shrink dispersion estimates toward a fitted curve (borrowing information across genes)
4. Fit the NB model and perform hypothesis testing
5. Apply Benjamini-Hochberg correction for multiple testing

### Simplified DE with scipy

For educational purposes, we will implement a basic DE analysis using a Mann-Whitney U test (non-parametric) on normalized counts. This is a simplification; production analyses should use DESeq2 or edgeR.

In [ ]:
def simple_de_analysis(count_df, sample_info, condition_col='condition',
                       control='Control', treatment='Treatment'):
    """
    Perform simplified differential expression analysis.
    Uses Mann-Whitney U test on normalized counts.
    """
    # Normalize
    sf = deseq2_size_factors(count_df)
    norm_counts = count_df.div(sf, axis=1)
    
    ctrl_samples = sample_info[sample_info[condition_col] == control].index
    treat_samples = sample_info[sample_info[condition_col] == treatment].index
    
    results = []
    for gene in count_df.index:
        ctrl_vals = norm_counts.loc[gene, ctrl_samples].values.astype(float)
        treat_vals = norm_counts.loc[gene, treat_samples].values.astype(float)
        
        # Mean expression
        ctrl_mean = ctrl_vals.mean()
        treat_mean = treat_vals.mean()
        
        # Log2 fold change (add pseudocount to avoid log(0))
        log2fc = np.log2((treat_mean + 1) / (ctrl_mean + 1))
        
        # Mean of normalized counts (for MA plot)
        base_mean = (ctrl_mean + treat_mean) / 2
        
        # Statistical test
        if ctrl_vals.std() == 0 and treat_vals.std() == 0:
            pval = 1.0
        else:
            _, pval = stats.mannwhitneyu(ctrl_vals, treat_vals,
                                        alternative='two-sided')
        
        results.append({
            'gene': gene,
            'baseMean': base_mean,
            'log2FoldChange': log2fc,
            'pvalue': pval
        })
    
    results_df = pd.DataFrame(results).set_index('gene')
    
    # Multiple testing correction (Benjamini-Hochberg)
    pvals = results_df['pvalue'].values
    n = len(pvals)
    sorted_idx = np.argsort(pvals)
    sorted_pvals = pvals[sorted_idx]
    
    adjusted = np.zeros(n)
    adjusted[sorted_idx[-1]] = sorted_pvals[-1]
    for i in range(n - 2, -1, -1):
        adjusted[sorted_idx[i]] = min(
            sorted_pvals[i] * n / (i + 1),
            adjusted[sorted_idx[i + 1]]
        )
    adjusted = np.minimum(adjusted, 1.0)
    
    results_df['padj'] = adjusted
    results_df = results_df.sort_values('pvalue')
    
    return results_df

# Run DE analysis
de_results = simple_de_analysis(counts_df, sample_info)

print(f"Total genes tested: {len(de_results)}")
print(f"Significant (padj < 0.05): {(de_results['padj'] < 0.05).sum()}")
print(f"Significant (padj < 0.01): {(de_results['padj'] < 0.01).sum()}")
print(f"\nTop 10 DE genes:")
de_results.head(10)

## 12. Interpreting Results: Fold Change and Adjusted P-values

### Log2 Fold Change (LFC)

- **LFC > 0**: Gene is upregulated in treatment
- **LFC < 0**: Gene is downregulated in treatment
- **|LFC| > 1**: At least 2-fold change (a common threshold)

### Adjusted P-values

With thousands of genes tested, multiple testing correction is essential. Without it, at p < 0.05, we would expect 100 false positives among 2000 genes.

The **Benjamini-Hochberg procedure** controls the **false discovery rate (FDR)**: the expected proportion of false positives among all called significant genes.

- padj < 0.05 means we expect at most 5% of our significant results to be false positives
- padj < 0.01 is more stringent

## 13. Visualization: Volcano Plot

A volcano plot displays log2 fold change (x-axis) versus -log10(adjusted p-value) (y-axis). Significant genes with large effect sizes appear in the upper left and upper right corners.

In [ ]:
def volcano_plot(results, padj_threshold=0.05, lfc_threshold=1.0,
                 gene_info=None, ax=None):
    """
    Create a volcano plot from DE results.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 7))
    
    # Classify genes
    sig_up = (results['padj'] < padj_threshold) & (results['log2FoldChange'] > lfc_threshold)
    sig_down = (results['padj'] < padj_threshold) & (results['log2FoldChange'] < -lfc_threshold)
    not_sig = ~(sig_up | sig_down)
    
    neg_log10_padj = -np.log10(results['padj'].clip(lower=1e-50))
    
    # Plot non-significant first
    ax.scatter(results.loc[not_sig, 'log2FoldChange'], neg_log10_padj[not_sig],
               c='gray', alpha=0.4, s=10, label=f'Not significant ({not_sig.sum()})')
    ax.scatter(results.loc[sig_up, 'log2FoldChange'], neg_log10_padj[sig_up],
               c='firebrick', alpha=0.7, s=15, label=f'Up ({sig_up.sum()})')
    ax.scatter(results.loc[sig_down, 'log2FoldChange'], neg_log10_padj[sig_down],
               c='steelblue', alpha=0.7, s=15, label=f'Down ({sig_down.sum()})')
    
    # Thresholds
    ax.axhline(-np.log10(padj_threshold), color='black', linestyle='--', alpha=0.3)
    ax.axvline(lfc_threshold, color='black', linestyle='--', alpha=0.3)
    ax.axvline(-lfc_threshold, color='black', linestyle='--', alpha=0.3)
    
    ax.set_xlabel('log2 Fold Change')
    ax.set_ylabel('-log10(adjusted p-value)')
    ax.set_title('Volcano Plot')
    ax.legend(loc='upper right')
    
    # Label top genes
    top_genes = results.nsmallest(5, 'padj')
    for gene in top_genes.index:
        ax.annotate(gene,
                    (results.loc[gene, 'log2FoldChange'],
                     -np.log10(max(results.loc[gene, 'padj'], 1e-50))),
                    fontsize=7, ha='center', va='bottom')
    
    return ax

volcano_plot(de_results, gene_info=gene_info)
plt.tight_layout()
plt.show()

## 14. Visualization: MA Plot

An MA plot shows average expression (x-axis) versus log2 fold change (y-axis). It reveals whether the fold change depends on expression level, and whether normalization was effective (the cloud should be centered at LFC = 0).

In [ ]:
def ma_plot(results, padj_threshold=0.05, ax=None):
    """Create an MA plot from DE results."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))
    
    sig = results['padj'] < padj_threshold
    not_sig = ~sig
    
    ax.scatter(np.log10(results.loc[not_sig, 'baseMean'] + 1),
               results.loc[not_sig, 'log2FoldChange'],
               c='gray', alpha=0.3, s=10, label=f'Not significant ({not_sig.sum()})')
    ax.scatter(np.log10(results.loc[sig, 'baseMean'] + 1),
               results.loc[sig, 'log2FoldChange'],
               c='firebrick', alpha=0.5, s=15, label=f'Significant ({sig.sum()})')
    
    ax.axhline(0, color='black', linestyle='-', alpha=0.5)
    ax.set_xlabel('log10(Mean Expression)')
    ax.set_ylabel('log2 Fold Change')
    ax.set_title('MA Plot')
    ax.legend()
    
    return ax

ma_plot(de_results)
plt.tight_layout()
plt.show()

## 15. Visualization: Heatmap of Top DE Genes

In [ ]:
# Select top 30 DE genes by adjusted p-value
top_de = de_results[de_results['padj'] < 0.05].nsmallest(30, 'padj')

if len(top_de) > 0:
    # Z-score normalize across samples for visualization
    log_counts = np.log2(normalized_counts.loc[top_de.index] + 1)
    z_scores = log_counts.subtract(log_counts.mean(axis=1), axis=0).div(
        log_counts.std(axis=1), axis=0)
    
    # Hierarchical clustering of genes
    gene_linkage = linkage(z_scores, method='ward')
    gene_order = dendrogram(gene_linkage, no_plot=True)['leaves']
    z_scores_ordered = z_scores.iloc[gene_order]
    
    fig, ax = plt.subplots(figsize=(8, 10))
    im = ax.imshow(z_scores_ordered, cmap='RdBu_r', aspect='auto',
                   vmin=-2, vmax=2)
    
    ax.set_xticks(range(len(z_scores.columns)))
    ax.set_xticklabels(z_scores.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(z_scores_ordered)))
    ax.set_yticklabels(z_scores_ordered.index, fontsize=7)
    ax.set_title('Top DE Genes (z-score normalized)')
    
    plt.colorbar(im, ax=ax, shrink=0.5, label='Z-score')
    plt.tight_layout()
    plt.show()
else:
    print("No significant DE genes found at padj < 0.05.")

## 16. Comparing to Ground Truth

Since we simulated our data with known DE genes, we can evaluate how well our simple analysis recovered them.

In [ ]:
# Compare detected DE genes with ground truth
padj_threshold = 0.05
lfc_threshold = 0.5

detected = set(de_results[
    (de_results['padj'] < padj_threshold) &
    (de_results['log2FoldChange'].abs() > lfc_threshold)
].index)

true_positives_set = set(gene_info[gene_info['true_de']].index)

tp = len(detected & true_positives_set)
fp = len(detected - true_positives_set)
fn = len(true_positives_set - detected)
tn = len(set(gene_info.index) - detected - true_positives_set)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("DE Detection Performance:")
print(f"  True positives:  {tp}")
print(f"  False positives: {fp}")
print(f"  False negatives: {fn}")
print(f"  True negatives:  {tn}")
print(f"\n  Precision: {precision:.3f}")
print(f"  Recall:    {recall:.3f}")
print(f"  F1 score:  {f1:.3f}")
print(f"\nNote: Our simplified Mann-Whitney approach has lower power than DESeq2,")
print(f"especially with small sample sizes. DESeq2 borrows information across")
print(f"genes to estimate dispersion, which greatly improves detection.")

## 17. Gene Set Enrichment Analysis (GSEA)

After identifying DE genes, the next step is to understand their biological meaning. Are the DE genes enriched for specific pathways, biological processes, or molecular functions?

### Two Approaches

**Over-Representation Analysis (ORA)**:
- Input: a list of DE genes
- Test: Is any gene set over-represented among DE genes compared to chance? (Fisher's exact test / hypergeometric test)
- Limitation: depends on arbitrary threshold for defining "DE genes"

**Gene Set Enrichment Analysis (GSEA)**:
- Input: all genes ranked by a metric (e.g., signed log10 p-value)
- Test: Do genes in a particular gene set tend to cluster at the top or bottom of the ranked list?
- Advantage: uses information from all genes, not just those passing a threshold

### Common Gene Set Databases

| Database | Content |
|----------|--------|
| **GO** (Gene Ontology) | Biological Process, Molecular Function, Cellular Component |
| **KEGG** | Metabolic and signaling pathways |
| **Reactome** | Curated biological pathways |
| **MSigDB** | Hallmark, positional, motif, and other gene sets |

### Tools

- **R**: clusterProfiler, fgsea, enrichR
- **Python**: gseapy, goatools
- **Web**: DAVID, Enrichr, g:Profiler, PANTHER

### Practical: Simple Over-Representation Analysis

Let's demonstrate the concept with simulated gene sets.

In [ ]:
from scipy.stats import fisher_exact

# Simulate some gene sets (pathways)
all_genes = set(counts_df.index)
de_genes = set(de_results[
    (de_results['padj'] < 0.05) &
    (de_results['log2FoldChange'].abs() > 0.5)
].index)

np.random.seed(123)
gene_list = list(all_genes)

# Create gene sets: some enriched with true DE genes, some random
true_de_list = list(true_positives_set)
gene_sets = {
    'Immune_Response': set(np.random.choice(true_de_list, min(30, len(true_de_list)), replace=False)) |
                       set(np.random.choice(gene_list, 20, replace=False)),
    'Cell_Cycle': set(np.random.choice(true_de_list, min(25, len(true_de_list)), replace=False)) |
                  set(np.random.choice(gene_list, 25, replace=False)),
    'Metabolism': set(np.random.choice(gene_list, 60, replace=False)),
    'Transport': set(np.random.choice(gene_list, 45, replace=False)),
    'Signaling': set(np.random.choice(true_de_list, min(15, len(true_de_list)), replace=False)) |
                 set(np.random.choice(gene_list, 35, replace=False)),
}

print(f"DE genes: {len(de_genes)}")
print(f"Total genes: {len(all_genes)}\n")

# Over-representation analysis with Fisher's exact test
ora_results = []
for pathway, pathway_genes in gene_sets.items():
    overlap = de_genes & pathway_genes
    
    # 2x2 contingency table
    #                   In pathway    Not in pathway
    # DE                a             b
    # Not DE            c             d
    a = len(overlap)
    b = len(de_genes - pathway_genes)
    c = len(pathway_genes - de_genes)
    d = len(all_genes - de_genes - pathway_genes)
    
    odds_ratio, pval = fisher_exact([[a, b], [c, d]], alternative='greater')
    
    ora_results.append({
        'Pathway': pathway,
        'Size': len(pathway_genes),
        'Overlap': a,
        'Odds Ratio': odds_ratio,
        'P-value': pval
    })

ora_df = pd.DataFrame(ora_results).sort_values('P-value')
print("Over-Representation Analysis Results:")
print(ora_df.to_string(index=False))

In [ ]:
# Visualize ORA results as a dot plot
fig, ax = plt.subplots(figsize=(8, 5))

ora_sorted = ora_df.sort_values('P-value', ascending=False)

sizes = ora_sorted['Overlap'] * 20
colors = -np.log10(ora_sorted['P-value'])

scatter = ax.scatter(ora_sorted['Odds Ratio'], range(len(ora_sorted)),
                     s=sizes, c=colors, cmap='RdYlBu_r',
                     edgecolors='black', linewidth=0.5)

ax.set_yticks(range(len(ora_sorted)))
ax.set_yticklabels(ora_sorted['Pathway'])
ax.set_xlabel('Odds Ratio')
ax.set_title('Pathway Over-Representation Analysis')
ax.axvline(1, color='gray', linestyle='--', alpha=0.5)

plt.colorbar(scatter, ax=ax, label='-log10(p-value)', shrink=0.7)

plt.tight_layout()
plt.show()

## 18. Using DESeq2 in Practice (R Integration)

For production RNA-seq analysis, use DESeq2 (R) or pydeseq2 (Python). Here is the standard R workflow:

```r
library(DESeq2)

# Create DESeqDataSet from count matrix and sample info
dds <- DESeqDataSetFromMatrix(
    countData = count_matrix,
    colData = sample_info,
    design = ~ condition
)

# Run the full DESeq2 pipeline
dds <- DESeq(dds)

# Extract results
res <- results(dds, contrast = c("condition", "Treatment", "Control"))

# Apply LFC shrinkage (recommended for visualization and ranking)
res_shrunk <- lfcShrink(dds, coef = "condition_Treatment_vs_Control",
                        type = "apeglm")

# Summary
summary(res)

# Volcano plot
plotMA(res)
```

### pydeseq2 (Python Alternative)

```python
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

dds = DeseqDataSet(
    counts=count_matrix,
    metadata=sample_info,
    design_factors="condition"
)
dds.deseq2()

stat_res = DeseqStats(dds, contrast=["condition", "Treatment", "Control"])
stat_res.summary()
results_df = stat_res.results_df
```

## Differential Expression in R: DESeq2 and edgeR

While this course focuses on Python, the gold-standard tools for differential expression are R packages: DESeq2 and edgeR. Here we show the standard workflow so you can read and adapt R-based analysis pipelines.

In [ ]:
# The standard DESeq2 workflow in R:
deseq2_workflow = """
# Load count matrix and sample metadata
library("DESeq2")
expr.matrix <- as.matrix(read.table("counts.txt", header=TRUE, row.names=1, sep="\\t"))
design <- data.frame(
    row.names = colnames(expr.matrix),
    condition = c("control", "control", "treated", "treated")
)

# Create DESeq2 dataset and run analysis
dds <- DESeqDataSetFromMatrix(countData = expr.matrix, colData = design, design = ~ condition)
dds <- DESeq(dds)
res <- results(dds)

# Filter significant genes
sig <- subset(res, padj < 0.05)
up <- subset(sig, log2FoldChange > 1)    # upregulated
down <- subset(sig, log2FoldChange < -1)  # downregulated

# Visualization
plotMA(dds, ylim=c(-2,2), main="DESeq2 MA Plot")
plotDispEsts(dds)
"""
print(deseq2_workflow)

The edgeR workflow follows a similar pattern:
1. Create DGEList from count matrix
2. Filter low-count genes (filterByExpr)
3. Normalize library sizes (calcNormFactors)
4. Estimate dispersion (estimateDisp)
5. Test for differential expression (exactTest or glmQLFTest)
6. Filter by FDR and fold change

Both tools use negative binomial models for count data. DESeq2 is more conservative (fewer false positives); edgeR is more powerful (fewer false negatives). Many papers use both and report the intersection.

In [ ]:
from scipy import stats
import numpy as np

# Simplified differential expression with Python (for understanding)
# In practice, use DESeq2/edgeR for proper normalization and dispersion estimation
np.random.seed(42)
n_genes = 1000
n_samples = 4  # 2 control + 2 treated

# Simulate count data
control = np.random.negative_binomial(n=5, p=0.01, size=(n_genes, 2))
treated = control.copy()
# Make 50 genes differentially expressed (2x upregulated)
de_genes = np.arange(50)
treated[de_genes] = np.random.negative_binomial(n=5, p=0.005, size=(50, 2))

# Simple fold change + t-test (educational only -- NOT production quality)
log2fc = np.log2(treated.mean(axis=1) + 1) - np.log2(control.mean(axis=1) + 1)
pvalues = np.array([stats.ttest_ind(control[i], treated[i]).pvalue for i in range(n_genes)])

# Volcano plot
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 6))
colors = ['red' if (abs(log2fc[i]) > 1 and pvalues[i] < 0.05) else 'gray' for i in range(n_genes)]
plt.scatter(log2fc, -np.log10(pvalues + 1e-300), c=colors, alpha=0.5, s=10)
plt.xlabel('log2 Fold Change')
plt.ylabel('-log10(p-value)')
plt.title('Volcano Plot (simplified)')
plt.axhline(-np.log10(0.05), color='blue', linestyle='--', alpha=0.5)
plt.axvline(1, color='blue', linestyle='--', alpha=0.5)
plt.axvline(-1, color='blue', linestyle='--', alpha=0.5)
plt.show()
print(f"Significant DE genes: {sum((abs(log2fc) > 1) & (pvalues < 0.05))}")

## 19. Complete Mini-Workflow

Let's put together a complete analysis pipeline from count matrix to biological interpretation, in a single cohesive workflow.

In [ ]:
# ============================================================
# STEP 1: Generate data (in real analysis, you load count matrix)
# ============================================================
print("=" * 60)
print("STEP 1: Load/Generate Count Data")
print("=" * 60)

workflow_counts, workflow_meta, workflow_truth = simulate_rnaseq_counts(
    n_genes=3000, n_samples_per_group=5, n_de_genes=300,
    fold_change_range=(1.5, 5.0)
)

print(f"Loaded count matrix: {workflow_counts.shape[0]} genes x {workflow_counts.shape[1]} samples")
print(f"Conditions: {workflow_meta['condition'].value_counts().to_dict()}")
print(f"Total counts range: {workflow_counts.sum(axis=0).min():.0f} - {workflow_counts.sum(axis=0).max():.0f}")

In [ ]:
# ============================================================
# STEP 2: Quality filtering
# ============================================================
print("=" * 60)
print("STEP 2: Filter Low-Count Genes")
print("=" * 60)

# Keep genes with at least 10 counts in at least 3 samples
min_count = 10
min_samples = 3
keep = (workflow_counts >= min_count).sum(axis=1) >= min_samples
filtered_counts = workflow_counts[keep]

print(f"Before filtering: {workflow_counts.shape[0]} genes")
print(f"After filtering:  {filtered_counts.shape[0]} genes")
print(f"Removed: {(~keep).sum()} low-count genes")

In [ ]:
# ============================================================
# STEP 3: Normalize
# ============================================================
print("=" * 60)
print("STEP 3: Normalize (DESeq2 size factors)")
print("=" * 60)

wf_size_factors = deseq2_size_factors(filtered_counts)
wf_normalized = filtered_counts.div(wf_size_factors, axis=1)

print("Size factors:")
print(wf_size_factors.round(3))

In [ ]:
# ============================================================
# STEP 4: Exploratory analysis (PCA)
# ============================================================
print("=" * 60)
print("STEP 4: Exploratory Analysis")
print("=" * 60)

wf_log = np.log2(wf_normalized + 1)
wf_pca = PCA(n_components=2)
wf_pca_result = wf_pca.fit_transform(wf_log.T)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA
for condition, color, marker in [('Control', 'steelblue', 'o'), ('Treatment', 'coral', 's')]:
    mask = workflow_meta['condition'] == condition
    axes[0].scatter(wf_pca_result[mask, 0], wf_pca_result[mask, 1],
                    c=color, marker=marker, s=100, edgecolors='black',
                    label=condition, zorder=3)

axes[0].set_xlabel(f'PC1 ({wf_pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({wf_pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title('PCA: Sample Clustering')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Library sizes
lib_sizes_wf = filtered_counts.sum(axis=0)
bar_colors = ['steelblue' if c == 'Control' else 'coral' for c in workflow_meta['condition']]
axes[1].bar(range(len(lib_sizes_wf)), lib_sizes_wf / 1e6, color=bar_colors, edgecolor='black')
axes[1].set_xlabel('Sample')
axes[1].set_ylabel('Library Size (millions)')
axes[1].set_title('Library Sizes')
axes[1].set_xticks(range(len(lib_sizes_wf)))
axes[1].set_xticklabels(lib_sizes_wf.index, rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 5: Differential expression
# ============================================================
print("=" * 60)
print("STEP 5: Differential Expression Analysis")
print("=" * 60)

wf_de_results = simple_de_analysis(filtered_counts, workflow_meta)

sig_genes = wf_de_results[wf_de_results['padj'] < 0.05]
sig_up = sig_genes[sig_genes['log2FoldChange'] > 1]
sig_down = sig_genes[sig_genes['log2FoldChange'] < -1]

print(f"\nTotal genes tested: {len(wf_de_results)}")
print(f"Significant (padj < 0.05): {len(sig_genes)}")
print(f"  Upregulated (LFC > 1):   {len(sig_up)}")
print(f"  Downregulated (LFC < -1): {len(sig_down)}")

In [ ]:
# ============================================================
# STEP 6: Visualization
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

volcano_plot(wf_de_results, ax=axes[0])
axes[0].set_title('Volcano Plot')

ma_plot(wf_de_results, ax=axes[1])
axes[1].set_title('MA Plot')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 7: Summary report
# ============================================================
print("=" * 60)
print("ANALYSIS SUMMARY")
print("=" * 60)
print(f"")
print(f"Input: {workflow_counts.shape[0]} genes, {workflow_counts.shape[1]} samples")
print(f"After filtering: {filtered_counts.shape[0]} genes")
print(f"")
print(f"Differential Expression (padj < 0.05, |LFC| > 1):")
print(f"  Total significant: {len(sig_up) + len(sig_down)}")
print(f"  Upregulated:       {len(sig_up)}")
print(f"  Downregulated:     {len(sig_down)}")
print(f"")
print(f"Top 5 upregulated genes:")
top_up = wf_de_results[(wf_de_results['padj'] < 0.05) & (wf_de_results['log2FoldChange'] > 0)].nsmallest(5, 'padj')
for gene, row in top_up.iterrows():
    print(f"  {gene}: LFC={row['log2FoldChange']:.2f}, padj={row['padj']:.2e}")
print(f"")
print(f"Top 5 downregulated genes:")
top_down = wf_de_results[(wf_de_results['padj'] < 0.05) & (wf_de_results['log2FoldChange'] < 0)].nsmallest(5, 'padj')
for gene, row in top_down.iterrows():
    print(f"  {gene}: LFC={row['log2FoldChange']:.2f}, padj={row['padj']:.2e}")

## 20. Common Pitfalls

| Pitfall | Why it matters | Solution |
|---------|----------------|----------|
| Too few replicates | Low statistical power | Use at least 3 biological replicates per condition |
| Using TPM/FPKM for DE | These units are not suitable for statistical testing | Use raw counts with DESeq2/edgeR |
| No multiple testing correction | Many false positives | Always use adjusted p-values (BH/FDR) |
| Ignoring batch effects | Confounded results | Balance design; use batch correction (ComBat, limma) |
| Not filtering low counts | Increased multiple testing burden, unreliable estimates | Filter genes with very low counts before testing |
| Confusing biological and technical replicates | Inflated significance | Biological replicates = independent samples |
| Cherry-picking genes | Confirmation bias | Report genome-wide results, use gene set analysis |

## Hierarchical Clustering and Heatmaps

After identifying differentially expressed genes, hierarchical clustering and heatmaps help visualize expression patterns across samples. Clustering both rows (genes) and columns (samples) reveals co-regulated gene groups and sample relationships.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist
import seaborn as sns

# Simulate gene expression count matrix (100 genes, 6 samples)
np.random.seed(42)
n_genes, n_control, n_treated = 100, 3, 3
n_samples = n_control + n_treated
gene_names = [f'Gene_{i:03d}' for i in range(n_genes)]
sample_names = [f'Control_{i+1}' for i in range(n_control)] + [f'Treated_{i+1}' for i in range(n_treated)]

# Base expression + noise
base = np.random.negative_binomial(5, 0.01, size=(n_genes, n_samples)).astype(float)
# Make 20 genes upregulated in treated
base[:20, n_control:] *= np.random.uniform(2, 4, size=(20, n_treated))
# Make 10 genes downregulated in treated
base[20:30, n_control:] *= np.random.uniform(0.1, 0.4, size=(10, n_treated))

# Log-transform for visualization
log_expr = np.log2(base + 1)

# Sample correlation and dendrogram
corr = np.corrcoef(log_expr.T)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(corr, xticklabels=sample_names, yticklabels=sample_names,
            annot=True, fmt='.2f', cmap='RdBu_r', center=0.5, ax=ax1)
ax1.set_title('Sample Correlation Matrix')

dist = pdist(log_expr.T, metric='correlation')
Z = linkage(dist, method='average')
dendrogram(Z, labels=sample_names, ax=ax2)
ax2.set_title('Sample Dendrogram')
ax2.set_ylabel('1 - correlation')
plt.tight_layout()
plt.show()


In [ ]:
# Heatmap of top DE genes (z-scored)
de_genes = log_expr[:30]  # first 30 are our DE genes
z_scored = (de_genes - de_genes.mean(axis=1, keepdims=True)) / de_genes.std(axis=1, keepdims=True)

g = sns.clustermap(z_scored,
                   xticklabels=sample_names,
                   yticklabels=[f'Gene_{i:03d}' for i in range(30)],
                   cmap='RdBu_r', center=0,
                   figsize=(8, 10),
                   row_cluster=True, col_cluster=True)
g.fig.suptitle('Clustered Heatmap of DE Genes (z-scored)', y=1.02)
plt.show()


The heatmap clearly separates control from treated samples (column dendrogram) and upregulated from downregulated genes (row dendrogram). This is the standard visualization in RNA-seq papers — R's ComplexHeatmap package produces publication-quality versions, but seaborn's clustermap is sufficient for exploratory analysis.

## Exercises

### Exercise 1: Normalization Comparison

Implement CPM (Counts Per Million) normalization and compare it to DESeq2 size factor normalization.

```python
def counts_to_cpm(count_matrix):
    """Convert raw counts to CPM."""
    # Your code here
    pass

# Compare: plot the distribution of log2(CPM+1) vs log2(DESeq2_normalized+1)
# for one sample. Are they similar?
```

In [ ]:
# Exercise 1: your code here


### Exercise 2: Effect of Replication

Re-run the simulation and DE analysis with 2, 3, 5, and 10 replicates per group. For each, record the number of true positives, false positives, and false negatives. Plot how precision and recall change with replicate number.

```python
replicate_numbers = [2, 3, 5, 10]
results_by_rep = {}

for n_rep in replicate_numbers:
    # Simulate with n_rep samples per group
    # Run DE analysis
    # Calculate precision and recall
    pass
```

In [ ]:
# Exercise 2: your code here


### Exercise 3: Volcano Plot Customization

Modify the volcano plot to:
1. Use different thresholds (padj < 0.01, |LFC| > 2)
2. Label the top 10 most significant genes
3. Color up-regulated genes in red and down-regulated in blue
4. Add a legend showing the count of genes in each category

In [ ]:
# Exercise 3: your code here


### Exercise 4: Sample Outlier Detection

Using the correlation heatmap and PCA plot approaches from Section 10:

1. Simulate a dataset where one sample is an outlier (mix in counts from the wrong condition)
2. Identify the outlier using PCA and correlation analysis
3. Remove the outlier and re-run DE analysis. How do the results change?

In [ ]:
# Exercise 4: your code here


### Exercise 5: Multi-Condition Design

Extend the simulation to include 3 conditions (Control, Treatment_A, Treatment_B). Each treatment has different DE genes. Run pairwise comparisons and create a Venn-diagram-style summary showing which genes are shared vs unique across comparisons.

Hint: You can represent overlaps as a table:

| Comparison | Unique DE | Shared with other |
|------------|-----------|-------------------|
| A vs Control | ? | ? |
| B vs Control | ? | ? |
| A vs B | ? | ? |

In [ ]:
# Exercise 5: your code here


### Exercise 6: Implement TMM Normalization

Implement the TMM (Trimmed Mean of M-values) normalization used by edgeR:

1. Choose a reference sample (e.g., the one with library size closest to the mean)
2. For each sample, compute M-values (log2 fold change from reference) and A-values (average log expression)
3. Trim the top and bottom 30% of M-values and top and bottom 5% of A-values
4. The TMM factor is the weighted mean of the remaining M-values

Compare TMM normalization to DESeq2 size factors.

In [ ]:
# Exercise 6: your code here


---

## Resources

### Essential Reading
- [RNA-seq analysis is easy as 1-2-3 with limma, Glimma and edgeR](https://f1000research.com/articles/5-1408) -- excellent workflow article
- [DESeq2 vignette](https://bioconductor.org/packages/release/bioc/vignettes/DESeq2/inst/doc/DESeq2.html) -- official DESeq2 documentation
- [Love et al., 2014](https://genomebiology.biomedcentral.com/articles/10.1186/s13059-014-0550-8) -- DESeq2 paper

### Tools
- [STAR](https://github.com/alexdobin/STAR) -- splice-aware aligner
- [Salmon](https://combine-lab.github.io/salmon/) -- alignment-free quantification
- [featureCounts](https://subread.sourceforge.net/) -- read counting
- [pydeseq2](https://github.com/owkin/PyDESeq2) -- Python implementation of DESeq2
- [gseapy](https://gseapy.readthedocs.io/) -- Python GSEA implementation

### Quality Control
- [FastQC](https://www.bioinformatics.babraham.ac.uk/projects/fastqc/)
- [MultiQC](https://multiqc.info/) -- aggregate QC reports
- [Trim Galore](https://www.bioinformatics.babraham.ac.uk/projects/trim_galore/) -- adapter/quality trimming

---

[← Previous: Variant Calling and SNP Analysis](../../Tier_3_Applied_Bioinformatics/02_Variant_Calling_and_SNP_Analysis/01_variant_calling_and_snp_analysis.ipynb) | [Next: Microbial Diversity Analysis →](../../Tier_3_Applied_Bioinformatics/04_Microbial_Diversity/01_microbial_diversity.ipynb)